In [ ]:
"""
I-BEAM 2-VARIABLE GP: b (web thickness) and H (web height)
- GP with alpha=1e-3, log-space target
- Marginal (averaged) 1D plots, global 1D/2D slices, a few named slices
- EI and UCB recommendations at 5 explore/exploit levels (no fantasy)
"""

import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

BOUNDS = {'b': (1.0, 8.0), 'H': (12.0, 23.0)}
PARAM_KEYS = list(BOUNDS.keys())
PARAM_NAMES = ['b_web (mm)', 'H_web (mm)']
NOISE_LEVEL = 1e-3

DATA_URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
            "core-me-data-science-activities-public/main/data/I_beam_data_2var.csv")

# ==========================================
# DATA
# ==========================================
def load_data():
    import urllib.request, io
    print("Downloading data...")
    with urllib.request.urlopen(DATA_URL) as r:
        df = pd.read_csv(io.StringIO(r.read().decode('utf-8')))
    df = df[['b_web_mm', 'H_web_mm', 'Str/w N/g']].dropna()
    df.columns = ['b', 'H', 'str_w']
    print(f"Loaded {len(df)} beams, Str/w: [{df['str_w'].min():.2f}, {df['str_w'].max():.2f}]")
    return df

# ==========================================
# NORMALIZE / DENORMALIZE
# ==========================================
def normalize(X):
    X_n = np.copy(X).astype(float)
    for i, k in enumerate(PARAM_KEYS):
        lo, hi = BOUNDS[k]
        X_n[:, i] = (X[:, i] - lo) / (hi - lo)
    return X_n

def denormalize(X_n):
    X = np.copy(X_n)
    for i, k in enumerate(PARAM_KEYS):
        lo, hi = BOUNDS[k]
        X[:, i] = X_n[:, i] * (hi - lo) + lo
    return X

# ==========================================
# GP
# ==========================================
def train_gp(X_2d, y):
    X_norm = normalize(X_2d)
    y_log = np.log(y)
    y_mean = np.mean(y_log)
    y_cent = y_log - y_mean
    kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(
        length_scale=[0.5, 0.5], length_scale_bounds=(0.1, 3.0), nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=25,
                                   alpha=NOISE_LEVEL, normalize_y=False)
    gp.fit(X_norm, y_cent)
    return gp, X_norm, y_cent, y_mean

# ==========================================
# ACQUISITION FUNCTIONS
# ==========================================
def ei_acq(gp, X_norm, y_best, xi):
    mu, sigma = gp.predict(X_norm, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    imp = mu - y_best - xi
    Z = imp / sigma
    return imp * norm.cdf(Z) + sigma * norm.pdf(Z), mu, sigma

def ucb_acq(gp, X_norm, kappa):
    mu, sigma = gp.predict(X_norm, return_std=True)
    return mu + kappa * np.maximum(sigma, 1e-9), mu, sigma

def generate_candidates(n=5000):
    b = np.random.uniform(*BOUNDS['b'], size=n)
    H = np.random.uniform(*BOUNDS['H'], size=n)
    return np.column_stack([b, H])

def recommend(gp, y_cent, y_mean, n_rec=5, n_cand=5000):
    """Return n_rec recommendations for 5 EI and 5 UCB explore/exploit levels."""
    xi_vals  = [0.0001, 0.001, 0.005, 0.01, 0.02]
    kappa_vals = [0.2, 0.5, 1.0, 1.5, 2.0]
    labels_ei = [f"xi={v}" for v in xi_vals]
    labels_ucb = [f"kappa={v}" for v in kappa_vals]
    y_best = y_cent.max()
    cand = generate_candidates(n_cand)
    cand_norm = normalize(cand)

    rows = []
    for xi, lab in zip(xi_vals, labels_ei):
        acq, mu, sig = ei_acq(gp, cand_norm, y_best, xi)
        top = np.argsort(acq)[-n_rec:][::-1]
        for rank, idx in enumerate(top, 1):
            rows.append({
                'Method': 'EI', 'Setting': lab,
                'Rank': rank,
                'b': round(cand[idx, 0], 3), 'H': round(cand[idx, 1], 3),
                'Pred_str_w': round(np.exp(mu[idx] + y_mean), 2),
                'sigma': round(sig[idx], 4),
            })
    for kappa, lab in zip(kappa_vals, labels_ucb):
        acq, mu, sig = ucb_acq(gp, cand_norm, kappa)
        top = np.argsort(acq)[-n_rec:][::-1]
        for rank, idx in enumerate(top, 1):
            rows.append({
                'Method': 'UCB', 'Setting': lab,
                'Rank': rank,
                'b': round(cand[idx, 0], 3), 'H': round(cand[idx, 1], 3),
                'Pred_str_w': round(np.exp(mu[idx] + y_mean), 2),
                'sigma': round(sig[idx], 4),
            })
    return pd.DataFrame(rows)

# ==========================================
# PLOTTING HELPERS
# ==========================================
def _uncertainty_bands(ax, x, mu_r, epistemic, y_mean_scalar):
    aleatory = mu_r * np.sqrt(NOISE_LEVEL)
    total = np.sqrt(epistemic**2 + aleatory**2)
    ax.fill_between(x, mu_r - 2*total, mu_r + 2*total,
                    alpha=0.25, color='orange', label='+-2s aleatory')
    ax.fill_between(x, mu_r - 2*epistemic, mu_r + 2*epistemic,
                    alpha=0.3, color='blue', label='+-2s epistemic')

def get_point_sizes(X_norm, slice_pt):
    d = np.linalg.norm(X_norm - slice_pt, axis=1)
    sizes = np.where(d < 0.15, 250,
            np.where(d < 0.30, 180,
            np.where(d < 0.50, 120,
            np.where(d < 0.75, 70, 30))))
    colors = np.where(d < 0.15, 0.9,
             np.where(d < 0.30, 0.7,
             np.where(d < 0.50, 0.5,
             np.where(d < 0.75, 0.3, 0.1))))
    return sizes, d, colors

# ==========================================
# MARGINAL 1D (AVERAGED)
# ==========================================
def plot_1d_marginal(gp, X_norm, y, y_mean, n_samples=200):
    print("Marginal 1D plots...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('2-Var I-Beam GP: Marginal 1D (Averaged)', fontsize=14, weight='bold')

    for dim in range(2):
        ax = axes[dim]
        x_norm = np.linspace(0, 1, 100)
        mu_all, sig_all = [], []
        for xv in x_norm:
            samp = np.random.rand(n_samples, 2)
            samp[:, dim] = xv
            mu, sig = gp.predict(samp, return_std=True)
            mu_all.append(np.mean(mu))
            sig_all.append(np.mean(sig))
        mu_avg = np.array(mu_all)
        sig_avg = np.array(sig_all)
        mu_r = np.exp(mu_avg + y_mean)
        epistemic = mu_r * sig_avg
        lo, hi = BOUNDS[PARAM_KEYS[dim]]
        x_vals = x_norm * (hi - lo) + lo

        ax.plot(x_vals, mu_r, 'b-', lw=2, label='Marginal mean')
        _uncertainty_bands(ax, x_vals, mu_r, epistemic, y_mean)

        X_den = denormalize(X_norm)
        ax.scatter(X_den[:, dim], y, c='red', s=50, alpha=0.6, ec='black', lw=1, zorder=5, label='Data')
        ax.set_xlabel(PARAM_NAMES[dim]); ax.set_ylabel('Str/w (N/g)')
        ax.set_title(PARAM_NAMES[dim], weight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('ibeam2v_marginal_1d.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# GLOBAL 1D SLICES
# ==========================================
def plot_1d_global(gp, X_norm, y, y_mean):
    print("Global 1D slices...")
    slice_pt = np.mean(X_norm, axis=0)
    slice_real = denormalize(slice_pt.reshape(1, -1))[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Global 1D Slices (b={slice_real[0]:.2f}, H={slice_real[1]:.2f})',
                 fontsize=14, weight='bold')
    for dim in range(2):
        ax = axes[dim]
        X_test = np.tile(slice_pt, (100, 1))
        X_test[:, dim] = np.linspace(0, 1, 100)
        mu, sig = gp.predict(X_test, return_std=True)
        mu_r = np.exp(mu + y_mean)
        epistemic = mu_r * sig
        x_vals = denormalize(X_test)[:, dim]
        ax.plot(x_vals, mu_r, 'b-', lw=2, label='Mean')
        _uncertainty_bands(ax, x_vals, mu_r, epistemic, y_mean)
        X_den = denormalize(X_norm)
        ax.scatter(X_den[:, dim], y, c='red', s=50, alpha=0.6, ec='black', lw=1, zorder=5, label='Data')
        ax.set_xlabel(PARAM_NAMES[dim]); ax.set_ylabel('Str/w (N/g)')
        ax.set_title(PARAM_NAMES[dim], weight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('ibeam2v_global_1d.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# GLOBAL 2D CONTOUR
# ==========================================
def plot_2d_global(gp, X_norm, y, y_mean):
    print("Global 2D contour...")
    res = 60
    gx, gy = np.meshgrid(np.linspace(0, 1, res), np.linspace(0, 1, res))
    X_test = np.column_stack([gx.ravel(), gy.ravel()])
    mu, sig = gp.predict(X_test, return_std=True)
    mu_r = np.exp(mu + y_mean).reshape(res, res)
    sig_r = sig.reshape(res, res)

    ext = [BOUNDS['b'][0], BOUNDS['b'][1], BOUNDS['H'][0], BOUNDS['H'][1]]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('2-Var GP: Mean and Uncertainty', fontsize=14, weight='bold')

    X_den = denormalize(X_norm)
    im1 = axes[0].imshow(mu_r, origin='lower', extent=ext, aspect='auto', cmap='viridis')
    axes[0].contour(mu_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
    axes[0].scatter(X_den[:, 0], X_den[:, 1], c='red', s=80, ec='white', lw=2, zorder=10, label='Data')
    axes[0].set_xlabel('b (mm)'); axes[0].set_ylabel('H (mm)')
    axes[0].set_title('Mean Str/w', weight='bold'); axes[0].legend()
    plt.colorbar(im1, ax=axes[0], label='Str/w (N/g)')

    im2 = axes[1].imshow(sig_r, origin='lower', extent=ext, aspect='auto', cmap='hot')
    axes[1].contour(sig_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
    axes[1].scatter(X_den[:, 0], X_den[:, 1], c='cyan', s=80, ec='white', lw=2, zorder=10, label='Data')
    axes[1].set_xlabel('b (mm)'); axes[1].set_ylabel('H (mm)')
    axes[1].set_title('Uncertainty (sigma)', weight='bold'); axes[1].legend()
    plt.colorbar(im2, ax=axes[1], label='Std Dev')

    plt.tight_layout()
    plt.savefig('ibeam2v_global_2d.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# SLICE 1D
# ==========================================
def plot_slice_1d(gp, X_norm, y, y_mean, slice_pt_norm, name, idx):
    slice_real = denormalize(slice_pt_norm.reshape(1, -1))[0]
    sizes, dists, colors = get_point_sizes(X_norm, slice_pt_norm)
    on_slice = dists < 0.1

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Slice {idx}: {name} (b={slice_real[0]:.2f}, H={slice_real[1]:.2f})',
                 fontsize=14, weight='bold')
    for dim in range(2):
        ax = axes[dim]
        X_test = np.tile(slice_pt_norm, (100, 1))
        X_test[:, dim] = np.linspace(0, 1, 100)
        mu, sig = gp.predict(X_test, return_std=True)
        mu_r = np.exp(mu + y_mean)
        epistemic = mu_r * sig
        x_vals = denormalize(X_test)[:, dim]
        ax.plot(x_vals, mu_r, 'b-', lw=2, label='Mean')
        _uncertainty_bands(ax, x_vals, mu_r, epistemic, y_mean)
        X_den = denormalize(X_norm)
        ax.scatter(X_den[:, dim], y, s=sizes, c=colors, cmap='Reds',
                  vmin=0, vmax=1, alpha=0.6, ec='black', lw=0.5, zorder=5, label='Data')
        if np.any(on_slice):
            ax.scatter(X_den[on_slice, dim], y[on_slice], s=400, marker='*',
                      c='yellow', ec='black', lw=2, zorder=10, label='On slice')
        ax.set_xlabel(PARAM_NAMES[dim]); ax.set_ylabel('Str/w (N/g)')
        ax.set_title(PARAM_NAMES[dim], weight='bold')
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    clean = name.replace('#','num').replace(' ','_').replace('(','').replace(')','').replace('=','').replace('/','_')
    plt.savefig(f'ibeam2v_slice_{idx:02d}_{clean}_1d.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# SLICE 2D
# ==========================================
def plot_slice_2d(gp, X_norm, y, y_mean, slice_pt_norm, name, idx):
    slice_real = denormalize(slice_pt_norm.reshape(1, -1))[0]
    sizes, dists, colors = get_point_sizes(X_norm, slice_pt_norm)
    on_slice = dists < 0.1

    res = 60
    gx, gy = np.meshgrid(np.linspace(0, 1, res), np.linspace(0, 1, res))
    X_test = np.column_stack([gx.ravel(), gy.ravel()])
    mu, sig = gp.predict(X_test, return_std=True)
    mu_r = np.exp(mu + y_mean).reshape(res, res)
    sig_r = sig.reshape(res, res)
    ext = [BOUNDS['b'][0], BOUNDS['b'][1], BOUNDS['H'][0], BOUNDS['H'][1]]

    X_den = denormalize(X_norm)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'Slice {idx}: {name} (b={slice_real[0]:.2f}, H={slice_real[1]:.2f})',
                 fontsize=14, weight='bold')

    im1 = axes[0].imshow(mu_r, origin='lower', extent=ext, aspect='auto', cmap='viridis')
    axes[0].contour(mu_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
    axes[0].scatter(X_den[:, 0], X_den[:, 1], s=sizes, c=colors, cmap='Reds',
                   vmin=0, vmax=1, alpha=0.6, ec='white', lw=1, zorder=5)
    if np.any(on_slice):
        axes[0].scatter(X_den[on_slice, 0], X_den[on_slice, 1], s=400, marker='*',
                       c='yellow', ec='black', lw=2, zorder=10)
    axes[0].plot(slice_real[0], slice_real[1], 'r+', ms=20, mew=3, zorder=11)
    axes[0].set_xlabel('b (mm)'); axes[0].set_ylabel('H (mm)')
    axes[0].set_title('Mean', weight='bold')
    plt.colorbar(im1, ax=axes[0], label='Str/w')

    im2 = axes[1].imshow(sig_r, origin='lower', extent=ext, aspect='auto', cmap='hot')
    axes[1].contour(sig_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
    axes[1].scatter(X_den[:, 0], X_den[:, 1], s=sizes, c=colors, cmap='Blues',
                   vmin=0, vmax=1, alpha=0.6, ec='white', lw=1, zorder=5)
    if np.any(on_slice):
        axes[1].scatter(X_den[on_slice, 0], X_den[on_slice, 1], s=400, marker='*',
                       c='yellow', ec='black', lw=2, zorder=10)
    axes[1].plot(slice_real[0], slice_real[1], 'r+', ms=20, mew=3, zorder=11)
    axes[1].set_xlabel('b (mm)'); axes[1].set_ylabel('H (mm)')
    axes[1].set_title('Uncertainty', weight='bold')
    plt.colorbar(im2, ax=axes[1], label='Std Dev')

    plt.tight_layout()
    clean = name.replace('#','num').replace(' ','_').replace('(','').replace(')','').replace('=','').replace('/','_')
    plt.savefig(f'ibeam2v_slice_{idx:02d}_{clean}_2d.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# RECOMMENDATION PLOT
# ==========================================
def plot_recommendations(gp, X_norm, y, y_mean, rec_df):
    """Plot recommendations on the 2D contour."""
    res = 60
    gx, gy = np.meshgrid(np.linspace(0, 1, res), np.linspace(0, 1, res))
    X_test = np.column_stack([gx.ravel(), gy.ravel()])
    mu, _ = gp.predict(X_test, return_std=True)
    mu_r = np.exp(mu + y_mean).reshape(res, res)
    ext = [BOUNDS['b'][0], BOUNDS['b'][1], BOUNDS['H'][0], BOUNDS['H'][1]]
    X_den = denormalize(X_norm)

    for method in ['EI', 'UCB']:
        sub = rec_df[rec_df['Method'] == method]
        settings = sub['Setting'].unique()

        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.imshow(mu_r, origin='lower', extent=ext, aspect='auto', cmap='viridis', alpha=0.7)
        ax.contour(mu_r, origin='lower', extent=ext, colors='white', alpha=0.3, levels=8)
        ax.scatter(X_den[:, 0], X_den[:, 1], c='red', s=60, ec='white', lw=1.5, zorder=10, label='Data')

        markers = ['o', 's', 'D', '^', 'v']
        cmap = plt.cm.cool(np.linspace(0.1, 0.9, len(settings)))
        for i, (setting, color) in enumerate(zip(settings, cmap)):
            ss = sub[sub['Setting'] == setting]
            exploit_label = 'exploit' if i == 0 else ('explore' if i == len(settings)-1 else '')
            label = f"{setting}" + (f" ({exploit_label})" if exploit_label else "")
            ax.scatter(ss['b'], ss['H'], c=[color]*len(ss), s=120, marker=markers[i % len(markers)],
                      ec='black', lw=1.5, zorder=11, label=label)

        ax.set_xlabel('b (mm)'); ax.set_ylabel('H (mm)')
        ax.set_title(f'{method} Recommendations (5 explore/exploit levels, top 5 each)',
                     fontsize=12, weight='bold')
        ax.legend(fontsize=8, loc='best')
        ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
        plt.tight_layout()
        plt.savefig(f'ibeam2v_recs_{method.lower()}.png', dpi=150, bbox_inches='tight')
        plt.show()

# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)

    df = load_data()
    X_2d = df[['b', 'H']].values
    y = df['str_w'].values

    print(f"\nTraining GP on {len(y)} beams (2D, alpha={NOISE_LEVEL})...")
    gp, X_norm, y_cent, y_mean = train_gp(X_2d, y)
    scales = gp.kernel_.k2.length_scale
    print(f"ARD length scales: b={scales[0]:.3f}, H={scales[1]:.3f}")

    best_idx = np.argmax(y)
    print(f"Best beam: b={X_2d[best_idx,0]:.2f}, H={X_2d[best_idx,1]:.2f} -> Str/w={y[best_idx]:.2f}")

    # Recommendations
    print("\nGenerating recommendations (5 EI + 5 UCB levels, top 5 each)...")
    rec_df = recommend(gp, y_cent, y_mean)
    for method in ['EI', 'UCB']:
        print(f"\n{'='*60}\n{method} RECOMMENDATIONS\n{'='*60}")
        sub = rec_df[rec_df['Method'] == method]
        for setting in sub['Setting'].unique():
            ss = sub[sub['Setting'] == setting]
            print(f"\n  {setting}:")
            print(ss[['Rank', 'b', 'H', 'Pred_str_w', 'sigma']].to_string(index=False))

    # Plots
    plot_1d_marginal(gp, X_norm, y, y_mean)
    plot_1d_global(gp, X_norm, y, y_mean)
    plot_2d_global(gp, X_norm, y, y_mean)

    # Slices: top 3 beams + a few interesting points
    slices = []
    top3 = np.argsort(y)[-3:][::-1]
    for i, idx in enumerate(top3):
        slices.append((X_2d[idx], f"Top {i+1} Str_w={y[idx]:.2f}"))
    slices.append((np.array([2.0, 14.0]), "Thin web, short"))
    slices.append((np.array([6.0, 20.0]), "Thick web, tall"))

    for si, (pt, name) in enumerate(slices):
        pt_norm = normalize(pt.reshape(1, -1))[0]
        print(f"\nSlice {si+1}: {name}")
        plot_slice_1d(gp, X_norm, y, y_mean, pt_norm, name, si+1)
        plot_slice_2d(gp, X_norm, y, y_mean, pt_norm, name, si+1)

    plot_recommendations(gp, X_norm, y, y_mean, rec_df)
    print("\nDone!")
